# Dispersão de sal em canal inclinado – processamento dos dados

**Disciplina:** Monitoramento Ambiental com Sensores e Microcontroladores (UFPR, 2026-2)
**Grupo Plant/Harrison** – sensor do grupo: **ESP-D68785 (posição A)**, sempre comparado aos outros quatro.

Todo o processamento deste notebook é **externo ao sensor** (feito no PC a partir do arquivo do data-logger):

1. Leitura do `sensor_data.csv` e reamostragem em grade de 1 s
2. Remoção de falhas de leitura e do trecho com sonda fora d'água
3. Estatística do período de referência (antes da 1ª injeção) e intercalibração
4. Separação das 5 injeções, linha de base, média móvel e momentos temporais
5. Velocidade média *U* e coeficiente de dispersão *D_L* entre duas seções (momentos e *routing* de Fischer)
6. Análise de sensibilidade, figuras e tabelas
7. Verificação com o *Dispersion Analyzer*, o pacote da disciplina (baixado do GitHub)
8. Curva de calibração da sonda A (ESP-D68785), a partir do `calibração.txt`
9. Movimento da pluma: diagrama distância × tempo, perfis ao longo do canal e previsão da curva em E
10. Concentração de sal e cloreto adicionado, comparados com o limite da Resolução CONAMA 357/2005

**Como usar:** `Ambiente de execução > Executar tudo`. Os dois arquivos de dados (`sensor_data.csv` e `calibração.txt`) são baixados do Google Drive pelos links da seção 1. No final, o notebook baixa um `.zip` com as figuras (PNG e PDF) e as tabelas (CSV). As seções 1 e 8 precisam de acesso à internet.

In [ ]:
from pathlib import Path
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy.optimize import least_squares
from IPython.display import display

# integração trapezoidal compatível com NumPy 1.x e 2.x
trapz = getattr(np, "trapezoid", None) or np.trapz

PASTA = Path("resultados")
PASTA_FIG = PASTA / "figuras"
PASTA_TAB = PASTA / "tabelas"
PASTA_FIG.mkdir(parents=True, exist_ok=True)
PASTA_TAB.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)

## 1. Arquivo de dados

Os arquivos são baixados do Google Drive pelos links abaixo (compartilhados como "qualquer pessoa com o link"):

- `sensor_data.csv`: leituras das 5 placas no ensaio em canal (09/11/2026), enviadas por Wi-Fi;
- `calibração.txt`: calibração da sonda A em concentração (05/10/2026), gravada no cartão microSD.

Se o download do `sensor_data.csv` falhar, o notebook pede o upload do arquivo. Sem o arquivo de calibração, as seções 9 a 11 são puladas.

In [ ]:
import re, subprocess, sys

LINK_SENSOR_DATA = "https://drive.google.com/file/d/1PtcX1IXsd1fZRJ0pMwMyCOU1-S4wnPna/view?usp=sharing"
LINK_CALIBRACAO = "https://drive.google.com/file/d/1EpEQQ0zdYj_DXlX-7dQee2hb-oIOOSrf/view?usp=sharing"
ARQUIVO = "sensor_data.csv"
ARQUIVO_CALIB = "calibracao.txt"


def baixar_drive(link, destino):
    """Baixa um arquivo compartilhado do Google Drive (se ainda não estiver na pasta)."""
    if os.path.exists(destino) or not link:
        return
    try:
        import gdown                                # já vem instalado no Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    m = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", link)    # ID do arquivo no link do Drive
    try:
        gdown.download(id=m.group(1) or m.group(2), output=destino, quiet=False)
    except Exception as erro:
        print(f"Não foi possível baixar {destino} do Drive: {erro}")


baixar_drive(LINK_SENSOR_DATA, ARQUIVO)
baixar_drive(LINK_CALIBRACAO, ARQUIVO_CALIB)

if not os.path.exists(ARQUIVO):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Coloque o arquivo {ARQUIVO} na mesma pasta do notebook.")
    print("Selecione o arquivo sensor_data.csv")
    enviado = files.upload()
    ARQUIVO = next(iter(enviado))

print("Arquivo de dados:", ARQUIVO)
print("Arquivo de calibração:", ARQUIVO_CALIB if os.path.exists(ARQUIVO_CALIB) else "não encontrado")

## 2. Metadados do experimento

Valores da planilha compartilhada da turma. **Edite aqui** se alguma distância, inclinação ou horário for corrigido.

- Sensor A a 91,15 cm do início do canal, como o sensor B (seção 1).
- A posição **D** (ESP-104D45) foi associada pela distância anotada (147 cm) e pela ordem de chegada dos pulsos.
- Sensor **E** (ESP-C9C537) a 233 cm do início do canal e 11 cm do fundo; entra nos trechos C→E e A→E.

In [ ]:
DATA = "2026-11-09"

# posição -> placa, distância d do início do canal (cm), altura h acima do fundo (cm)
SENSORES = {
    "A": dict(board="ESP-D68785", d=91.15, h=18.0, grupo="Harrison"),
    "B": dict(board="ESP-848B36", d=91.15, h=6.5, grupo="McCartney"),
    "C": dict(board="ESP-4EE77C", d=146.0, h=21.5, grupo="Star"),
    "D": dict(board="ESP-104D45", d=147.0, h=8.6, grupo="Bonham"),
    "E": dict(board="ESP-C9C537", d=233.0, h=11.0, grupo="Richards"),
}
NOSSO = "A"
BOARD2POS = {v["board"]: k for k, v in SENSORES.items()}

# horário anotado da injeção, inclinação (graus), Δh do manômetro (cm), fim da janela de análise
EXPERIMENTOS = pd.DataFrame(
    [
        (1, "16:08:00", 0.0, 24.5, "16:12:00"),
        (2, "16:19:00", 1.0, 22.0, "16:24:30"),
        (3, "16:29:00", 2.0, 22.0, "16:32:00"),   # Δh "igual ao do exp. 2"
        (4, "16:41:00", 2.5, 22.0, "16:47:00"),
        (5, "16:52:00", 2.3, 25.3, "16:55:30"),
    ],
    columns=["exp", "injecao", "inclinacao_graus", "dh_manometro_cm", "fim"],
)
for c in ["injecao", "fim"]:
    EXPERIMENTOS[c] = pd.to_datetime(DATA + " " + EXPERIMENTOS[c])

MASSA_SAL_G = 128.0 * 0.125                         # 128 kg/m³ x 125 mL = 16 g de NaCl por injeção
FIM_REFERENCIA = pd.Timestamp(DATA + " 16:08:00")   # antes da 1ª injeção
FORA_DAGUA_C = pd.Timestamp(DATA + " 16:55:40")     # placa C retirada da água
LIMIAR_PULSO = 0.05                                 # fração do pico que delimita o pulso
T_BUSCA_PICO = 150.0                                # s após a injeção: limite da busca do pico
SUAVIZACAO_S = 5                                    # janela da média móvel (s)
ENSAIO_FOCO = 2                                     # ensaio analisado em detalhe (16:19, canal a 1°)

# Venturi da alimentação do canal (manômetro de mercúrio) e nível d'água anotado junto aos sensores
VENTURI = dict(D1=0.024, D2=0.0136, Cd=1.0, rho_rel=13.6)        # m; Cd não informado, adotado 1
NIVEL_CM = {1: [22.5], 2: [21.0], 3: [18.7, 23.5], 4: [12.0], 5: [16.4]}   # ensaio 3: dois valores anotados
LARGURA_CANAL = 0.145                               # m
NU_AGUA = 1.03e-6                                   # m²/s, água a 19 °C

EXPERIMENTOS

In [ ]:
# Estilo das figuras: o sensor do grupo (A) em destaque, os demais em traço fino
CORES = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a", "D": "#eda100", "E": "#e87ba4"}
TINTA, TINTA2, GRADE = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10,
    "axes.edgecolor": TINTA2, "axes.labelcolor": TINTA, "xtick.color": TINTA2,
    "ytick.color": TINTA2, "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "lines.linewidth": 1.0,
})
VIRGULA = FuncFormatter(lambda x, _: f"{x:g}".replace(".", ","))
FMT_HORA = mdates.DateFormatter("%H:%M")


def rotulo(p):
    extra = ""
    return f"{p} – {SENSORES[p]['board']}{extra}"


def estilo(p):
    if p == NOSSO:
        return dict(color=CORES[p], lw=1.5, zorder=5)
    return dict(color=CORES[p], lw=0.9, alpha=0.9, zorder=3)


def salvar(fig, nome):
    """Salva em PNG e PDF na pasta de figuras e mostra na tela."""
    fig.savefig(PASTA_FIG / f"{nome}.png")
    fig.savefig(PASTA_FIG / f"{nome}.pdf")
    plt.show()


def caixas(ax, dados, **kw):
    """Box-plot por sensor, com as cores do notebook."""
    bp = ax.boxplot(dados, widths=0.5, patch_artist=True,
                    medianprops=dict(color=TINTA, lw=1.2), **kw)
    ax.set_xticks(range(1, len(SENSORES) + 1))
    ax.set_xticklabels(list(SENSORES))
    for patch, p in zip(bp["boxes"], SENSORES):
        patch.set_facecolor(CORES[p])
        patch.set_alpha(0.7 if p == NOSSO else 0.35)
        patch.set_edgecolor(CORES[p])
    return bp

## 3. Leitura, reamostragem e limpeza

- **Grade de 1 s:** cada placa grava cerca de uma leitura por segundo, mas com pequenas lacunas; as séries são colocadas numa grade comum.
- **Falhas de leitura:** o sal só pode *aumentar* a condutividade, então leituras abaixo de 75 % da mediana móvel de 61 s são tratadas como falha e removidas.
- **Sonda fora d'água:** o trecho final da placa C (≈ 50 µS/cm) é descartado.
- Lacunas de até 5 s são interpoladas linearmente.

In [ ]:
def carregar(caminho):
    d = pd.read_csv(caminho)
    d["time"] = pd.to_datetime(d["time"], dayfirst=True)
    d["pos"] = d["board_id"].map(BOARD2POS)
    return d


def grade_1s(d, var):
    p = d.pivot_table(index="time", columns="pos", values=var, aggfunc="mean")
    idx = pd.date_range(p.index.min(), p.index.max(), freq="1s")
    return p.reindex(idx)[list(SENSORES)]


def limpar(cond):
    bruto = cond.copy()
    limpo = cond.copy()
    mask = pd.DataFrame(False, index=cond.index, columns=cond.columns)
    for p in cond:
        med = cond[p].rolling(61, center=True, min_periods=10).median()
        m = cond[p] < 0.75 * med
        mask[p] = m
        limpo.loc[m, p] = np.nan
    fora = limpo.index >= FORA_DAGUA_C
    limpo.loc[fora, "C"] = np.nan
    mask.loc[fora, "C"] = True
    limpo = limpo.interpolate(limit=5, limit_area="inside")
    return bruto, limpo, mask


d = carregar(ARQUIVO)
bruto, limpo, falhas = limpar(grade_1s(d, "conductivity_us_cm"))
temp = grade_1s(d, "temperature_c").interpolate(limit=5, limit_area="inside")

ordenado = d.sort_values("time").groupby("pos")["time"]
resumo = d.groupby("pos").agg(
    placa=("board_id", "first"), n_leituras=("time", "size"),
    inicio=("time", "min"), fim=("time", "max"),
    T_min=("temperature_c", "min"), T_max=("temperature_c", "max"),
    CE_min=("conductivity_us_cm", "min"), CE_max=("conductivity_us_cm", "max"),
)
resumo["dt_medio_s"] = ordenado.apply(lambda s: s.diff().dt.total_seconds().mean())
resumo["lacunas_>2s"] = ordenado.apply(lambda s: int((s.diff().dt.total_seconds() > 2).sum()))
resumo["falhas_removidas"] = falhas.sum()
resumo.to_csv(PASTA_TAB / "resumo_registro.csv")
resumo

## 4. Período de referência (antes da 1ª injeção)

Com todos os sensores na mesma água, as diferenças entre eles são de calibração. O **fator de intercalibração** leva cada sensor à média dos cinco.

In [ ]:
ref = limpo.loc[:FIM_REFERENCIA - pd.Timedelta(seconds=10)]
est_ref = pd.DataFrame({
    "media": ref.mean(), "desvio": ref.std(), "mediana": ref.median(),
    "min": ref.min(), "max": ref.max(), "n": ref.count(),
})
est_ref["CV_%"] = 100 * est_ref["desvio"] / est_ref["media"]
est_ref["fator_intercal"] = ref.stack().mean() / est_ref["media"]
est_ref["T_media"] = temp.loc[ref.index].mean()
est_ref.round(3).to_csv(PASTA_TAB / "estatistica_referencia.csv")
est_ref.round(3)

## 5. Pulsos: linha de base, suavização e momentos temporais

Para cada injeção e cada sensor:

- **Linha de base** = mediana dos 60 s anteriores à injeção. Depois do pulso a condutividade volta a esse nível e só então sobe devagar (retorno do sal pela recirculação), por isso o pulso é medido em relação ao nível pré-injeção.
- **Média móvel** de 5 s sobre o excesso de condutividade ΔCE.
- **Janela do pulso:** do último ponto abaixo de 5 % do pico (antes dele) até a curva ficar 5 s abaixo desse limiar (depois dele).
- **Momentos:** área, centroide $\bar t$ e variância $\sigma_t^2$ da curva dentro da janela.

Como ΔCE é proporcional à concentração para soluções diluídas, os momentos normalizados não dependem da curva de calibração CE × concentração.

In [ ]:
def momentos(t, c):
    area = trapz(c, t)
    tm = trapz(t * c, t) / area
    var = trapz((t - tm) ** 2 * c, t) / area
    return area, tm, var


def analisar_pulso(serie, t_inj, t_fim, suav=SUAVIZACAO_S, limiar=LIMIAR_PULSO, n_abaixo=5):
    """Curva corrigida e estatísticas de um pulso (tempo em s desde a injeção)."""
    s = serie.loc[t_inj - pd.Timedelta(seconds=60):t_fim]
    t = (s.index - t_inj).total_seconds().values.astype(float)
    c = s.interpolate(limit_direction="both").values
    b0 = np.nanmedian(c[:60])     # nível antes da injeção
    b1 = np.nanmedian(c[-60:])    # nível no fim da janela (acúmulo de sal)
    dc = c - b0
    dc_s = pd.Series(dc).rolling(suav, center=True, min_periods=1).mean().values
    # pico procurado só até T_BUSCA_PICO após a injeção, para não confundir o pulso
    # com o retorno do sal pela recirculação (que chega a ~90 % do pico no sensor E)
    ate = int(np.searchsorted(t, T_BUSCA_PICO, side="right"))
    ip = int(np.nanargmax(dc_s[:ate]))
    pico = dc_s[ip]
    lim = limiar * pico
    # início: último ponto abaixo do limiar antes do pico
    abaixo = np.where(dc_s[:ip] < lim)[0]
    i0 = abaixo[-1] if len(abaixo) else 0
    # fim: curva n_abaixo segundos abaixo do limiar; se isso não ocorrer em 60 s
    # (retorno do sal já elevando o sinal), termina no mínimo da curva nesse intervalo
    lim_busca = min(ip + 60, len(dc_s) - n_abaixo)
    i1 = ip + int(np.nanargmin(dc_s[ip:lim_busca + 1]))
    for k in range(ip, lim_busca):
        if np.all(dc_s[k:k + n_abaixo] < lim):
            i1 = k
            break
    area, tm, var = momentos(t[i0:i1 + 1], np.clip(dc_s[i0:i1 + 1], 0, None))
    return dict(t=t, c=c, dc=dc, dc_s=dc_s, b0=b0, b1=b1, pico=pico, t_pico=t[ip],
                t_ini=t[i0], t_fim=t[i1], area=area, tm=tm, var=var)


def curva_janela(r, fator=1.0):
    """Curva suavizada, positiva e zerada fora da janela do pulso."""
    y = np.clip(r["dc_s"], 0, None) * fator
    y[(r["t"] < r["t_ini"]) | (r["t"] > r["t_fim"])] = 0.0
    return y


pulsos, linhas = {}, []
for _, e in EXPERIMENTOS.iterrows():
    for p in SENSORES:
        r = analisar_pulso(limpo[p], e.injecao, e.fim)
        pulsos[(e.exp, p)] = r
        linhas.append(dict(
            exp=e.exp, pos=p, placa=SENSORES[p]["board"],
            base_antes=r["b0"], base_depois=r["b1"], pico_dCE=r["pico"],
            t_pico_s=r["t_pico"], t_centroide_s=r["tm"], sigma_t_s=np.sqrt(r["var"]),
            area_uScm_s=r["area"],
            area_intercal=r["area"] * est_ref.loc[p, "fator_intercal"],
            duracao_s=r["t_fim"] - r["t_ini"],
        ))
tab_p = pd.DataFrame(linhas)
tab_p.round(2).to_csv(PASTA_TAB / "momentos_por_sensor.csv", index=False)
tab_p.round(2)

## 6. Velocidade média e coeficiente de dispersão entre duas seções

Seção 1: sensores A e B (d = 91,15 cm). Seção 2: sensores C e D (d ≈ 146 cm). Também são calculados os trechos até o sensor E (d = 233 cm): C→E e A→E.

**Método dos momentos**

$$U = \frac{\Delta x}{\bar t_2 - \bar t_1} \qquad D_L = \frac{U^2}{2}\,\frac{\sigma_{t,2}^2 - \sigma_{t,1}^2}{\bar t_2 - \bar t_1}$$

**Routing de Fischer** (resultado principal): a curva medida na seção 1 é propagada até a seção 2 pela solução da equação de advecção–dispersão, e *U* e *D_L* são ajustados por mínimos quadrados à curva medida na seção 2:

$$c(x_2,t) = \int c(x_1,\tau)\,\frac{U}{\sqrt{4\pi D_L \bar t}}\,\exp\!\left[-\frac{U^2(\bar t - t + \tau)^2}{4 D_L \bar t}\right]d\tau, \qquad \bar t = \frac{\Delta x}{U}$$

O *routing* usa a forma inteira da curva e é menos sensível às caudas do que a diferença de variâncias, que pode até ficar negativa.

In [ ]:
def routing(t, c1, U, D, dx):
    tbar = dx / U
    dt = t[1] - t[0]
    tau = t[:, None]    # instante na seção 1
    tt = t[None, :]     # instante na seção 2
    k = U / np.sqrt(4 * np.pi * D * tbar) * np.exp(-(U ** 2) * (tbar - tt + tau) ** 2 / (4 * D * tbar))
    return (c1[:, None] * k).sum(axis=0) * dt


def ajuste_routing(t, c1, c2, dx, U0, D0=1e-3):
    n1, n2 = c1 / trapz(c1, t), c2 / trapz(c2, t)   # curvas normalizadas pela área

    def resid(par):
        return routing(t, n1, par[0], 10 ** par[1], dx) - n2

    sol = least_squares(resid, [np.clip(U0, 0.01, 0.4), np.log10(D0)],
                        bounds=([0.005, -6], [0.5, -1]))
    U, D = sol.x[0], 10 ** sol.x[1]
    prev = routing(t, n1, U, D, dx)
    # R² só no trecho do pulso (curva medida ou prevista acima de 1 % do pico);
    # na janela inteira os zeros antes e depois do pulso inflariam o R²
    m = (n2 > 0.01 * n2.max()) | (prev > 0.01 * prev.max())
    r2 = 1 - np.sum((prev[m] - n2[m]) ** 2) / np.sum((n2[m] - n2[m].mean()) ** 2)
    return U, D, r2, n1, n2, prev


def resultado_secoes(t, c1, c2, dx):
    _, tm1, v1 = momentos(t, c1)
    _, tm2, v2 = momentos(t, c2)
    dtm = tm2 - tm1
    U = dx / dtm
    D = U ** 2 * (v2 - v1) / (2 * dtm)
    Ur, Dr, r2, n1, n2, prev = ajuste_routing(t, c1, c2, dx, U, max(D, 1e-4))
    return dict(dt_centroides_s=dtm, U_mom=U, D_mom=D, U_rout=Ur, D_rout=Dr, R2_rout=r2,
                var1_s2=v1, var2_s2=v2), (n1, n2, prev)


TRECHO_AC, TRECHO_BD, TRECHO_MEDIA = "A→C", "B→D", "Seção 1→2 (média A,B → C,D)"
TRECHO_CE, TRECHO_AE = "C→E", "A→E"          # trechos até o sensor E, a jusante (d = 233 cm)
TRECHOS = {TRECHO_AC: (["A"], ["C"]), TRECHO_BD: (["B"], ["D"]), TRECHO_MEDIA: (["A", "B"], ["C", "D"]),
           TRECHO_CE: (["C"], ["E"]), TRECHO_AE: (["A"], ["E"])}

fi = est_ref["fator_intercal"]
res, curvas_rout = [], {}
for _, e in EXPERIMENTOS.iterrows():
    t = pulsos[(e.exp, "A")]["t"]
    for nome, (s1, s2) in TRECHOS.items():
        c1 = np.mean([curva_janela(pulsos[(e.exp, p)], fi[p]) for p in s1], axis=0)
        c2 = np.mean([curva_janela(pulsos[(e.exp, p)], fi[p]) for p in s2], axis=0)
        dx = (np.mean([SENSORES[p]["d"] for p in s2]) - np.mean([SENSORES[p]["d"] for p in s1])) / 100
        out, crv = resultado_secoes(t, c1, c2, dx)
        curvas_rout[(e.exp, nome)] = (t, *crv)
        res.append(dict(exp=e.exp, inclinacao_graus=e.inclinacao_graus,
                        S0=np.tan(np.radians(e.inclinacao_graus)),
                        dh_manometro_cm=e.dh_manometro_cm, trecho=nome, dx_m=dx, **out))
tab_r = pd.DataFrame(res)
tab_r.round(10).to_csv(PASTA_TAB / "dispersao_duas_secoes.csv", index=False)

# visão resumida em cm/s e cm²/s
tab_r.assign(U_cm_s=tab_r.U_rout * 100, DL_cm2_s=tab_r.D_rout * 1e4, DL_momentos_cm2_s=tab_r.D_mom * 1e4)[
    ["exp", "inclinacao_graus", "trecho", "dt_centroides_s", "U_cm_s", "DL_cm2_s", "DL_momentos_cm2_s", "R2_rout"]
].round(2)

### Sensibilidade aos parâmetros de tratamento (trecho A→C)

O cálculo é repetido com limiar de 2, 5 e 10 % do pico e média móvel de 3, 5 e 7 s. A faixa resultante (mínimo e máximo) é usada como barra de incerteza na figura de *D_L*.

In [ ]:
dx_AC = (SENSORES["C"]["d"] - SENSORES["A"]["d"]) / 100
sens = []
for _, e in EXPERIMENTOS.iterrows():
    for lim in [0.02, 0.05, 0.10]:
        for sv in [3, 5, 7]:
            r1 = analisar_pulso(limpo["A"], e.injecao, e.fim, suav=sv, limiar=lim)
            r2 = analisar_pulso(limpo["C"], e.injecao, e.fim, suav=sv, limiar=lim)
            out, _ = resultado_secoes(r1["t"], curva_janela(r1), curva_janela(r2), dx_AC)
            sens.append(dict(exp=e.exp, limiar=lim, suavizacao_s=sv, **out))
tab_s = pd.DataFrame(sens)
tab_s.round(6).to_csv(PASTA_TAB / "sensibilidade_A_C.csv", index=False)
faixa = tab_s.groupby("exp").agg(
    D_mom_min=("D_mom", "min"), D_mom_max=("D_mom", "max"),
    D_rout_min=("D_rout", "min"), D_rout_max=("D_rout", "max"),
    U_rout_min=("U_rout", "min"), U_rout_max=("U_rout", "max"),
)
faixa.to_csv(PASTA_TAB / "sensibilidade_A_C_faixas.csv")
(faixa[["D_rout_min", "D_rout_max"]] * 1e4).round(1).rename(columns=lambda c: c + "_cm2_s")

### Velocidade pela continuidade (Venturi) e $D_L$ pela expressão de Fischer

Vazão pelo Venturi da alimentação ($D_1$ = 24 mm, $D_2$ = 13,6 mm, $C_d$ = 1, manômetro de mercúrio):

$$Q = C_d\,\frac{\pi D_2^2}{4}\sqrt{\frac{2 g \Delta h (\rho_{Hg}/\rho_w - 1)}{1 - (D_2/D_1)^4}}$$

Velocidade pela continuidade, $U = Q/(B H)$, com o nível d'água anotado em cada ensaio, e $D_L = 0{,}011\,U^2B^2/(H u_*)$ (Fischer), com $u_*$ de parede lisa: $u_* = U\sqrt{f/8}$, $f = 0{,}316\,Re^{-1/4}$, $Re = 4UR/\nu$. Comparação com o *routing* no trecho A→C.

In [ ]:
def vazao_venturi(dh_cm):
    """Vazão (m³/s) pelo Venturi, Δh do manômetro de mercúrio em cm."""
    v = VENTURI
    a2 = np.pi * v["D2"] ** 2 / 4
    return v["Cd"] * a2 * np.sqrt(2 * 9.81 * dh_cm / 100 * (v["rho_rel"] - 1) / (1 - (v["D2"] / v["D1"]) ** 4))


def dl_fischer(U, H, B=LARGURA_CANAL):
    """D_L (m²/s) pela expressão de Fischer, com u* de parede lisa (Blasius). U em m/s, H e B em m."""
    R = B * H / (B + 2 * H)
    Re = 4 * U * R / NU_AGUA
    f = 0.316 * Re ** -0.25
    u_est = U * np.sqrt(f / 8)
    return 0.011 * U ** 2 * B ** 2 / (H * u_est), u_est


cont = []
for _, e in EXPERIMENTOS.iterrows():
    Q = vazao_venturi(e.dh_manometro_cm)
    rr = tab_r[(tab_r.exp == e.exp) & (tab_r.trecho == TRECHO_AC)].iloc[0]
    for H in NIVEL_CM[e.exp]:
        U = Q / (LARGURA_CANAL * H / 100)
        D, u_est = dl_fischer(U, H / 100)
        cont.append(dict(exp=e.exp, Q_L_s=1000 * Q, H_cm=H, U_cont_cm_s=100 * U, U_rout_cm_s=100 * rr.U_rout,
                         dif_U_pct=100 * (rr.U_rout - U) / U, u_est_cm_s=100 * u_est,
                         DL_fischer_cm2_s=1e4 * D, DL_rout_cm2_s=1e4 * rr.D_rout, razao_fischer_rout=D / rr.D_rout))
continuidade = pd.DataFrame(cont)
continuidade.round(4).to_csv(PASTA_TAB / "continuidade_fischer.csv", index=False)
continuidade.round(2)

## 7. Figuras

Cada figura é mostrada aqui e salva em `resultados/figuras` (PNG a 300 dpi e PDF vetorial para o Overleaf).

As figuras F1, F3, F5, F6 e F7 mostram todos os ensaios. As figuras do transporte (F2b, F17, F9 e, na seção 10, F13 e F14) mostram só o ensaio definido em `ENSAIO_FOCO` (ensaio 2, canal a 1°), com o tempo contado a partir do instante de injeção estimado. A F16 (seção 11) volta a mostrar todos os ensaios.

In [ ]:
# instante de injeção estimado (s após o minuto anotado): extrapolação dos centroides de A e C até x = 0
T0 = {}
for e in EXPERIMENTOS.exp:
    tA, tC = pulsos[(e, "A")]["tm"], pulsos[(e, "C")]["tm"]
    U_AC = (SENSORES["C"]["d"] - SENSORES["A"]["d"]) / (tC - tA)          # cm/s
    T0[e] = float(round(tA - SENSORES["A"]["d"] / U_AC))
FOCO = EXPERIMENTOS.set_index("exp").loc[ENSAIO_FOCO]
NOME_FOCO = f"Ensaio {ENSAIO_FOCO} ({str(FOCO.inclinacao_graus).replace('.', ',')}°)"
print("Instante de injeção estimado (s após o minuto anotado):", T0)

In [ ]:
# F1 – panorama: série completa dos 5 sensores
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for p in SENSORES:
    ax.plot(limpo.index, limpo[p], label=rotulo(p), **estilo(p))
for _, e in EXPERIMENTOS.iterrows():
    ax.axvline(e.injecao, color=TINTA2, lw=0.6, ls="--", zorder=1)
    ax.text(e.injecao, 1.01, f"Exp. {e.exp}\n{str(e.inclinacao_graus).replace('.', ',')}°",
            transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7, color=TINTA2)
ax.set_ylabel("Condutividade elétrica (µS/cm)")
ax.set_xlabel("Horário (09/11/2026)")
ax.xaxis.set_major_formatter(FMT_HORA)
ax.set_ylim(0, None)
ax.legend(ncol=5, fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.2),
          handlelength=1.5, columnspacing=1.0)
salvar(fig, "F1_serie_temporal_todos")

In [ ]:
# F2 – sensor do grupo: dado bruto, limpo e suavizado
fig, axs = plt.subplots(2, 1, figsize=(7.2, 4.6), gridspec_kw={"height_ratios": [1.3, 1]})
ax = axs[0]
ax.plot(bruto.index, bruto[NOSSO], color="#a8a7a1", lw=0.7, label="Bruto (data-logger)")
ax.plot(limpo.index, limpo[NOSSO], color=CORES[NOSSO], lw=1.0, label="Após remoção de falhas")
m = falhas[NOSSO]
ax.scatter(bruto.index[m], bruto[NOSSO][m], s=10, color="#e34948", zorder=6, label="Falhas removidas")
for _, e in EXPERIMENTOS.iterrows():
    ax.axvline(e.injecao, color=TINTA2, lw=0.6, ls="--", zorder=1)
ax.set_ylabel("CE (µS/cm)")
ax.set_title(f"(a) Sensor {NOSSO} – {SENSORES[NOSSO]['board']}: registro completo", loc="left")
ax.xaxis.set_major_formatter(FMT_HORA)
ax.legend(fontsize=7, loc="upper left")

ax = axs[1]
r = pulsos[(ENSAIO_FOCO, NOSSO)]
tt = r["t"] - T0[ENSAIO_FOCO]                      # tempo desde a injeção estimada
ax.plot(tt, r["c"], color="#a8a7a1", lw=0.8, label="Limpo")
ax.axhline(r["b0"], color=TINTA2, lw=1.0, ls="--", label="Linha de base (pré-injeção)")
ax.plot(tt, r["b0"] + r["dc_s"], color=CORES[NOSSO], lw=1.6, label="Média móvel 5 s")
ax.axvspan(r["t_ini"] - T0[ENSAIO_FOCO], r["t_fim"] - T0[ENSAIO_FOCO], color=CORES[NOSSO], alpha=0.08,
           lw=0, label="Janela dos momentos")
ax.set_xlim(-40, 150)
ax.set_xlabel("Tempo desde a injeção (s)")
ax.set_ylabel("CE (µS/cm)")
ax.set_title(f"(b) Detalhe do pulso do {NOME_FOCO.lower()}", loc="left")
ax.legend(fontsize=7, ncol=1, loc="upper right")
fig.tight_layout()
salvar(fig, "F2_tratamento_sensor_A")

In [ ]:
# F3 – pulsos (ΔCE acima da linha de base) por experimento, todos os sensores
fig, axs = plt.subplots(1, 5, figsize=(7.4, 2.9))
for ax, (_, e) in zip(axs, EXPERIMENTOS.iterrows()):
    for p in SENSORES:
        r = pulsos[(e.exp, p)]
        ax.plot(r["t"], r["dc_s"], label=rotulo(p), **estilo(p))
    ax.set_xlim(0, 120)
    ax.set_title(f"Exp. {e.exp} ({str(e.inclinacao_graus).replace('.', ',')}°)", fontsize=8)
    ax.set_xlabel("t (s)")
    ax.tick_params(labelsize=7)
axs[0].set_ylabel("ΔCE acima da linha de base (µS/cm)")
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, ncol=5, fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.07))
fig.tight_layout(rect=(0, 0.05, 1, 1))
salvar(fig, "F3_pulsos_por_experimento")

In [ ]:
# F17 – pulsos do ensaio em foco nos 5 sensores: (a) excesso de CE; (b) curvas normalizadas e centroides
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.1))
for p in SENSORES:
    r = pulsos[(ENSAIO_FOCO, p)]
    tt = r["t"] - T0[ENSAIO_FOCO]
    st = estilo(p)
    axs[0].plot(tt, r["dc_s"], label=f"{rotulo(p)} (d = {SENSORES[p]['d']:g} cm)".replace(".", ","), **st)
    axs[1].plot(tt, curva_janela(r) / r["area"], **st)
    axs[1].axvline(r["tm"] - T0[ENSAIO_FOCO], color=st["color"], lw=0.7, ls=":")
for ax in axs:
    ax.set_xlim(0, 130)
    ax.set_xlabel("Tempo desde a injeção (s)")
    ax.yaxis.set_major_formatter(VIRGULA)
axs[0].set_ylabel("ΔCE acima da linha de base (µS/cm)")
axs[1].set_ylabel("Curva normalizada pela área (1/s)")
axs[0].set_title(f"(a) {NOME_FOCO}: excesso de condutividade", loc="left", fontsize=9)
axs[1].set_title("(b) Curvas normalizadas e centroides", loc="left", fontsize=9)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, ncol=3, fontsize=7, loc="lower center", bbox_to_anchor=(0.5, -0.1))
fig.tight_layout(rect=(0, 0.08, 1, 1))
salvar(fig, "F17_pulsos_ensaio_foco")

In [ ]:
# F5 – período de referência: box-plot e histogramas
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axs[0]
caixas(ax, [ref[p].dropna().values for p in SENSORES], flierprops=dict(markersize=2))
ax.set_ylabel("CE (µS/cm)")
ax.set_xlabel("Sensor")
ax.set_title("(a) Água antes da 1ª injeção", loc="left")

ax = axs[1]
bins = np.arange(np.floor(ref.min().min()), np.ceil(ref.max().max()) + 1, 0.5)
for p in SENSORES:
    ax.hist(ref[p].dropna(), bins=bins, histtype="step", label=p,
            color=CORES[p], lw=1.8 if p == NOSSO else 1.0)
ax.set_xlabel("CE (µS/cm)")
ax.set_ylabel("Frequência (leituras)")
ax.set_title("(b) Histogramas", loc="left")
ax.legend(fontsize=7, title="Sensor", title_fontsize=7)
fig.tight_layout()
salvar(fig, "F5_referencia_boxplot_histograma")

In [ ]:
# F6 – temperatura (média móvel de 30 s)
fig, ax = plt.subplots(figsize=(7.2, 2.6))
tsm = temp.rolling(30, center=True, min_periods=5).mean()
for p in SENSORES:
    ax.plot(tsm.index, tsm[p], label=rotulo(p), **estilo(p))
ax.set_ylabel("Temperatura (°C)")
ax.set_xlabel("Horário (09/11/2026)")
ax.xaxis.set_major_formatter(FMT_HORA)
ax.yaxis.set_major_formatter(VIRGULA)
ax.legend(ncol=5, fontsize=6.5, loc="upper center", bbox_to_anchor=(0.5, -0.28),
          handlelength=1.5, columnspacing=1.0)
salvar(fig, "F6_temperatura")

In [ ]:
# F7 – grandezas do pulso por sensor (cada ponto é um experimento)
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.6))
for ax, (col, tit) in zip(axs, [("t_centroide_s", "Tempo do centroide (s)"),
                                ("sigma_t_s", "Desvio-padrão temporal $\\sigma_t$ (s)"),
                                ("area_intercal", "Área ∫ΔCE dt (µS·s/cm)")]):
    caixas(ax, [tab_p.loc[tab_p.pos == p, col].values for p in SENSORES], showfliers=False)
    for i, p in enumerate(SENSORES, 1):
        v = tab_p.loc[tab_p.pos == p, col]
        ax.scatter(np.full(len(v), i), v, s=9, color=TINTA2, zorder=4)
    ax.set_title(tit, fontsize=8)
    ax.yaxis.set_major_formatter(VIRGULA)
    ax.set_xlabel("Sensor")
fig.tight_layout()
salvar(fig, "F7_boxplot_momentos")

In [ ]:
# F9 – ajuste por routing no ensaio em foco: trechos A→C, B→D e C→E
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.9))
for ax, (tr, p1, p2) in zip(axs, [(TRECHO_AC, "A", "C"), (TRECHO_BD, "B", "D"), (TRECHO_CE, "C", "E")]):
    t, n1, n2, prev = curvas_rout[(ENSAIO_FOCO, tr)]
    tt = t - T0[ENSAIO_FOCO]
    rr = tab_r[(tab_r.exp == ENSAIO_FOCO) & (tab_r.trecho == tr)].iloc[0]
    ax.plot(tt, n1, color=CORES[p1], lw=1.2, label=f"{p1} medido")
    ax.plot(tt, n2, color=CORES[p2], lw=1.6, label=f"{p2} medido")
    ax.plot(tt, prev, color=TINTA, lw=1.0, ls="--", label=f"{p2} previsto")
    ax.legend(fontsize=6.5, loc="upper right")
    ax.set_xlim(max(0, tt[n1 > 0].min() - 10), tt[n2 > 0].max() + 15)
    ax.set_title(f"{tr}: U = {100 * rr.U_rout:.1f} cm/s\n$D_L$ = {1e4 * rr.D_rout:.1f} cm²/s, "
                 f"R² = {rr.R2_rout:.2f}".replace(".", ","), fontsize=8)
    ax.set_xlabel("Tempo desde a injeção (s)")
    ax.tick_params(labelsize=7)
    ax.yaxis.set_major_formatter(VIRGULA)
axs[0].set_ylabel("Concentração normalizada (1/s)")
fig.tight_layout()
salvar(fig, "F9_routing_ensaio_foco")

## 8. Verificação com o *Dispersion Analyzer* (pacote da disciplina)

O pacote `dispersion-analyzer` (R. C. Bueno, 2026) é baixado do GitHub e aplicado às mesmas séries, já em grade de 1 s e sem as falhas de leitura. O resto (fundo, concentração em excesso, delimitação da pluma, Fischer, propagação, teste de mistura e conservação de massa) é feito pelas funções do pacote.

- **Calibração:** sem curva das sondas, usa-se o modo `nacl` (condutividade teórica do NaCl a 25 °C).
- **Recorte:** no fim de cada ensaio o sal que volta pela recirculação mantém o sinal alto e o pacote acusa curva truncada. Por isso cada sonda é recortada no fim do pulso principal (`t_max`), no instante delimitado na Seção 5.
- **Propagação:** o pacote fixa $U = \Delta x/\Delta\bar t$ e ajusta só $K$. São usadas a opção padrão (compara concentrações) e a de formas (`normalize_mass=True`). A opção `fit_U=True` não é usada: com $\Delta\bar t$ fixo, $U$ e $K$ entram no ajuste só pela razão $K/U^2$.
- **Métodos de um ponto** (pico, momentos, percentis, ajuste da ADE, Chatwin): aplicados mais abaixo, com o horário anotado e com o instante estimado; a tabela mostrada é a do ensaio em foco.

In [ ]:
import importlib.util, subprocess, sys

PASTA_DA = Path("dispersion-analyzer")
if not PASTA_DA.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/buenorc/dispersion-analyzer", str(PASTA_DA)], check=True)

spec = importlib.util.spec_from_file_location(
    "dispersion_analyzer", PASTA_DA / "__init__.py", submodule_search_locations=[str(PASTA_DA)])
da = importlib.util.module_from_spec(spec)
sys.modules["dispersion_analyzer"] = da
spec.loader.exec_module(da)
from dispersion_analyzer.calibration import Calibration
from dispersion_analyzer.project import Project, Section, Point, AnalysisOptions
from dispersion_analyzer.analysis import run_analysis
print(f"Dispersion Analyzer {da.__version__} — {da.__author__}")

# séries de cada sonda no formato do datalogger (tempo, temperatura, condutividade)
PASTA_DA_RES = PASTA / "dispersion_analyzer"
(PASTA_DA_RES / "series").mkdir(parents=True, exist_ok=True)
for p in SENSORES:
    pd.DataFrame({"data_hora": limpo.index.strftime("%d/%m/%Y %H:%M:%S"),
                  "temp(C)": temp[p].interpolate(limit_direction="both").round(2).values,
                  "cond(uS/cm)": limpo[p].round(2).values}).dropna(subset=["cond(uS/cm)"]).to_csv(
        PASTA_DA_RES / "series" / f"{p}.txt", sep="\t", index=False)

In [ ]:
CAL_DA = Calibration(name="NaCl teórico 25 °C", mode="nacl")
X_M = {p: SENSORES[p]["d"] / 100 for p in "ABCDE"}                # m
CONFIG_DA = {TRECHO_AC: {"S1": ["A"], "S2": ["C"]},
             TRECHO_BD: {"S1": ["B"], "S2": ["D"]},
             TRECHO_MEDIA: {"S1": ["A", "B"], "S2": ["C", "D"]},
             TRECHO_CE: {"S1": ["C"], "S2": ["E"]}}
VARIANTES_DA = {"padrão": False, "formas": True}                  # normalize_mass
PROF = {2: 0.21, 4: 0.12}                                          # m, nível d'água anotado

linhas_da, mistura_da, avisos_da = [], [], []
for _, e in EXPERIMENTOS.iterrows():
    for trecho, secs in CONFIG_DA.items():
        for var, norm in VARIANTES_DA.items():
            pontos = []
            for s, pts in secs.items():
                for p in pts:
                    t_max = analisar_pulso(limpo[p], e.injecao, e.fim)["t_fim"] + 2.0
                    pontos.append(Point(p, s, file=str(PASTA_DA_RES / "series" / f"{p}.txt"),
                                        calibration=CAL_DA.name,
                                        processing=dict(t_min=-60.0, t_max=float(t_max),
                                                        end_threshold=0.02, smooth=0)))
            prj = Project(name=f"Ensaio {e.exp}", M_injected=MASSA_SAL_G / 1000,
                          injection_time=e.injecao.strftime("%d/%m/%Y %H:%M:%S"),
                          calibrations=[CAL_DA], points=pontos,
                          sections=[Section(s, float(np.mean([X_M[p] for p in pts])), width=0.145,
                                            depth=PROF.get(e.exp)) for s, pts in secs.items()],
                          analysis=AnalysisOptions(single_moments=False, single_percentiles=False,
                                                   ade_fit=False, chatwin=False, use_tail=False,
                                                   routing_fit_U=False, routing_normalize_mass=norm))
            R = run_analysis(prj)
            par = R.pairs[("S1", "S2")]
            linhas_da.append(dict(exp=e.exp, trecho=trecho, variante=var,
                                  U_cm_s=100 * par["momentos"]["U"],
                                  K_fischer_cm2_s=1e4 * par["momentos"]["D"],
                                  K_propagacao_cm2_s=1e4 * par["routing"]["D"],
                                  NSE=par["routing"]["NSE"],
                                  area_2_sobre_1=par["mom2"]["M0"] / par["mom1"]["M0"]))
            if var == "padrão":
                if "Mistura lateral" in R.tables:
                    mistura_da.append(R.tables["Mistura lateral"].assign(exp=e.exp))
                avisos_da += [dict(exp=e.exp, trecho=trecho, aviso=w) for w in R.warnings]

tab_da = pd.DataFrame(linhas_da)
tab_da.round(4).to_csv(PASTA_TAB / "dispersion_analyzer_resumo.csv", index=False)
mistura_da = pd.concat(mistura_da)
mistura_da.to_csv(PASTA_TAB / "dispersion_analyzer_teste_mistura.csv", index=False)
pd.DataFrame(avisos_da).drop_duplicates().to_csv(PASTA_TAB / "dispersion_analyzer_avisos.csv", index=False)

# comparação no trecho A→C
ac_g = tab_r[tab_r.trecho == TRECHO_AC].set_index("exp")
ac_d = tab_da[tab_da.trecho == TRECHO_AC].pivot(index="exp", columns="variante")
comp_ac = pd.DataFrame({
    "U grupo (cm/s)": 100 * ac_g.U_rout, "K grupo routing": 1e4 * ac_g.D_rout,
    "K grupo momentos": 1e4 * ac_g.D_mom, "U DA (cm/s)": ac_d[("U_cm_s", "padrão")],
    "K DA Fischer": ac_d[("K_fischer_cm2_s", "padrão")],
    "K DA propagação padrão": ac_d[("K_propagacao_cm2_s", "padrão")],
    "K DA propagação formas": ac_d[("K_propagacao_cm2_s", "formas")],
    "NSE formas": ac_d[("NSE", "formas")]})
display(comp_ac.round(2))
display(mistura_da.iloc[:, [0, 2, 5, -1]].round(1))

### Métodos de um ponto (pico, momentos, percentis, ajuste da ADE, Chatwin)

O sal foi lançado no início do canal, então a distância de cada sensor ao lançamento é o seu $d$. O horário do lançamento foi anotado só até o minuto (hh:mm), e o tempo de viagem até o sensor A é de 13 a 34 s. Por isso são usadas duas referências:

- **horário anotado**, tomado como hh:mm:00;
- **instante estimado**, pela extrapolação dos centroides de A e C até $x = 0$: $t_0 = \bar t_A - x_A/U_{AC}$, com $U_{AC} = (x_C - x_A)/(\bar t_C - \bar t_A)$.

In [ ]:
tc = tab_p.pivot(index="exp", columns="pos", values="t_centroide_s")
linhas_1p, inst = [], []
for _, e in EXPERIMENTOS.iterrows():
    U_AC = (X_M["C"] - X_M["A"]) / (tc.loc[e.exp, "C"] - tc.loc[e.exp, "A"])
    # arredondado ao segundo: o pacote recebe o horário como texto hh:mm:ss
    t0 = float(round(tc.loc[e.exp, "A"] - X_M["A"] / U_AC))
    inst.append(dict(exp=e.exp, anotado=e.injecao.strftime("%H:%M"), t0_s=round(t0, 1),
                     estimado=(e.injecao + pd.Timedelta(seconds=t0)).strftime("%H:%M:%S")))
    for ref, desloc in [("horário anotado", 0.0), ("instante estimado", t0)]:
        pontos = [Point(p, p, file=str(PASTA_DA_RES / "series" / f"{p}.txt"), calibration=CAL_DA.name,
                        processing=dict(t_min=-60.0 - desloc, end_threshold=0.02, smooth=0,
                                        t_max=float(analisar_pulso(limpo[p], e.injecao, e.fim)["t_fim"] + 2 - desloc)))
                  for p in "ABCDE"]
        prj = Project(name=f"Ensaio {e.exp}", M_injected=MASSA_SAL_G / 1000, calibrations=[CAL_DA],
                      injection_time=(e.injecao + pd.Timedelta(seconds=desloc)).strftime("%d/%m/%Y %H:%M:%S"),
                      sections=[Section(p, X_M[p], width=0.145, depth=PROF.get(e.exp)) for p in "ABCDE"],
                      points=pontos,
                      analysis=AnalysisOptions(two_station_moments=False, routing=False, use_tail=False))
        for _, r in run_analysis(prj).tables["Resumo U e K"].iterrows():
            linhas_1p.append(dict(exp=e.exp, referencia=ref, sensor=r["Alvo"], metodo=r["Método"],
                                  U_cm_s=100 * r["U (m/s)"], K_cm2_s=1e4 * r["K (m²/s)"],
                                  NSE=r.get("NSE", np.nan)))
um_ponto = pd.DataFrame(linhas_1p)
um_ponto.round(4).to_csv(PASTA_TAB / "dispersion_analyzer_um_ponto.csv", index=False)
display(pd.DataFrame(inst))
ade = um_ponto[um_ponto.metodo.str.contains("ADE") & (um_ponto.exp == ENSAIO_FOCO)]
display(ade.pivot_table(index="sensor", columns="referencia", values=["U_cm_s", "K_cm2_s", "NSE"]).round(2))

## 9. Curva de calibração da sonda A (ESP-D68785)

Calibração feita em 05/10/2026 com o equipamento gravando no cartão microSD: sonda e DS18B20 em 1 L de água, leituras sem sal e depois de adições de NaCl até 0,1; 0,2; 0,3; 0,4; 0,6 e 0,8 g/L, por cerca de 1 min em cada concentração. A condutividade já vem compensada para 25 °C pelo código embarcado.

Leituras estáveis: as que ficam a até 2 % da mediana de cada concentração. A resposta da sonda não é linear, então a curva é uma lei de potência, $C = a\,(\kappa - \kappa_0)^b$, ajustada de 0,1 a 0,6 g/L (a leitura de 0,8 g/L não estabilizou). Para comparação, a figura mostra a condutividade teórica do NaCl a 25 °C (CRC Handbook, a mesma tabela do *Dispersion Analyzer*).

In [ ]:
import re
from scipy.optimize import curve_fit

TOL_ESTAVEL = 0.02      # leitura estável: até 2 % da mediana da concentração
C_MAX_AJUSTE = 600.0    # mg/L: maior concentração usada no ajuste

# condutividade molar do NaCl a 25 °C (S·cm²/mol), CRC Handbook
_NACL_C = np.array([0.0, 0.0005, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1])          # mol/L
_NACL_L = np.array([126.39, 124.44, 123.68, 120.59, 118.45, 115.70, 111.01, 106.69])


def kappa_teorico_nacl(c_mg_l):
    c = np.asarray(c_mg_l, float) / 1000 / 58.443
    return 1000 * np.interp(np.sqrt(c), np.sqrt(_NACL_C), _NACL_L) * c


def ler_calibracao(caminho):
    """Blocos separados por linha em branco, cada um com 'Concentração X' (g/L) e as leituras
    'data hora temperatura tensão condutividade'."""
    texto = open(caminho, encoding="utf-8", errors="replace").read()
    linhas = []
    for bloco in re.split(r"\n\s*\n", texto.strip()):
        m = re.search(r"Concentra\S*\s*:?\s*([\d.,]+)", bloco)
        if not m:
            continue
        C = 1000 * float(m.group(1).replace(",", "."))
        for ln in bloco.splitlines():
            p = ln.split()
            if len(p) >= 5 and "/" in p[0]:
                try:
                    T, V, k = (float(x.replace(",", ".")) for x in p[2:5])
                except ValueError:
                    continue
                linhas.append(dict(C_mg_L=C, hora=pd.to_datetime(p[0] + " " + p[1], dayfirst=True),
                                   T_C=T, V=V, kappa=k))
    return pd.DataFrame(linhas)


def lei_potencia(dk, a, b):
    return a * np.power(np.clip(dk, 0, None), b)


CAL_OK = os.path.exists(ARQUIVO_CALIB)
if not CAL_OK:
    print("Arquivo de calibração não encontrado: seções 9 a 11 puladas.")
else:
    cal_bruto = ler_calibracao(ARQUIVO_CALIB)
    linhas_cal = []
    for C, g in cal_bruto.groupby("C_mg_L"):
        med = g.kappa.median()
        est = g[(g.kappa - med).abs() <= TOL_ESTAVEL * med]
        linhas_cal.append(dict(C_mg_L=C, n_leituras=len(g), n_estaveis=len(est),
                               kappa_medio=est.kappa.mean(), desvio=est.kappa.std(),
                               T_media=g.T_C.mean(),
                               intervalo_s=g.hora.diff().dt.total_seconds().mean()))
    cal = pd.DataFrame(linhas_cal).set_index("C_mg_L")
    KAPPA0_CAL = cal.loc[0.0, "kappa_medio"]
    cal["excesso_kappa"] = cal.kappa_medio - KAPPA0_CAL
    cal["sensibilidade_uS_por_mg_L"] = cal.excesso_kappa / cal.index.to_series().replace(0, np.nan)
    cal["kappa_teorico_excesso"] = kappa_teorico_nacl(cal.index.values)
    cal["razao_medido_teorico"] = cal.excesso_kappa / cal.kappa_teorico_excesso
    usar = (cal.index > 0) & (cal.index <= C_MAX_AJUSTE)
    (A_CAL, B_CAL), _ = curve_fit(lei_potencia, cal.excesso_kappa[usar], cal.index[usar], p0=(1.0, 0.85))
    y = cal.index[usar].values
    prev = lei_potencia(cal.excesso_kappa[usar], A_CAL, B_CAL)
    R2_CAL = 1 - np.sum((y - prev) ** 2) / np.sum((y - y.mean()) ** 2)
    EP_CAL = np.sqrt(np.sum((y - prev) ** 2) / (len(y) - 2))
    DK_MAX_CAL = cal.excesso_kappa[usar].max()
    cal["C_ajustada"] = lei_potencia(cal.excesso_kappa, A_CAL, B_CAL)
    cal["usado_no_ajuste"] = usar
    cal.round(4).to_csv(PASTA_TAB / "calibracao_sonda_A.csv")
    print(f"C = {A_CAL:.4f} Δκ^{B_CAL:.4f}   R² = {R2_CAL:.4f}   erro-padrão = {EP_CAL:.1f} mg/L   κ0 = {KAPPA0_CAL:.1f} µS/cm")
    display(cal.round(2))


    def conc_nacl(dk):
        """NaCl em excesso (mg/L) a partir do excesso de condutividade (µS/cm), sonda A."""
        return lei_potencia(dk, A_CAL, B_CAL)

In [ ]:
# F12 – curva de calibração
if CAL_OK:
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    fora = ~usar & (cal.index > 0)
    ax.errorbar(cal.excesso_kappa[usar], cal.index[usar], xerr=cal.desvio[usar], fmt="o",
                color=CORES["A"], ms=5, capsize=2, label="Usado no ajuste")
    ax.errorbar(cal.excesso_kappa[fora], cal.index[fora], xerr=cal.desvio[fora], fmt="o", mfc="white",
                color=CORES["A"], ms=5, capsize=2, label="0,8 g/L (não estabilizou)")
    dd = np.linspace(0, cal.excesso_kappa.max() * 1.03, 200)
    ax.plot(dd, conc_nacl(dd), color=TINTA, lw=1.0, ls="--",
            label=("Ajuste: C = " + f"{A_CAL:.3f}".replace(".", ",") + " Δκ$^{" + f"{B_CAL:.3f}".replace(".", "{,}")
                   + "}$ (R² = " + f"{R2_CAL:.3f}".replace(".", ",") + ")"))
    cc = np.linspace(0, 800, 200)
    ax.plot(kappa_teorico_nacl(cc), cc, color=TINTA2, lw=1.0, ls=":", label="NaCl teórico, 25 °C")
    ax.set_xlabel("Excesso de condutividade Δκ (µS/cm)")
    ax.set_ylabel("NaCl em excesso (mg/L)")
    ax.xaxis.set_major_formatter(VIRGULA)
    ax.set_xlim(0, None)
    ax.set_ylim(0, None)
    ax.legend(fontsize=7, loc="upper left")
    salvar(fig, "F12_calibracao_sonda_A")

## 10. Movimento da pluma

As séries são convertidas em NaCl (mg/L) pela curva da sonda A. As outras sondas são levadas à escala de A pela razão entre os fatores de intercalibração, $\kappa_i f_i / f_A$ (supõe que diferem por um fator constante). O excesso é tomado em relação à água do canal antes da 1ª injeção, então inclui o sal acumulado pela recirculação.

Os tempos são contados a partir do instante de injeção estimado (extrapolação dos centroides de A e C até x = 0, arredondado ao segundo). Os perfis ao longo do canal e a previsão da curva em E usam o *routing* da curva de A com *U* e *D_L* do trecho A→C; como E não entra no ajuste, a previsão em E é uma validação do modelo. As tabelas trazem os cinco ensaios; as figuras F13 e F14 mostram o ensaio em foco.

In [ ]:
if CAL_OK:
    ESCALA_A = est_ref["fator_intercal"] / est_ref.loc["A", "fator_intercal"]
    KAPPA_REF_A = est_ref.loc["A", "media"]          # água do canal antes da 1ª injeção
    kappa_A = limpo * ESCALA_A                        # condutividade na escala da sonda A
    nacl_total = conc_nacl(kappa_A - KAPPA_REF_A)     # NaCl adicionado (mg/L)
    rot = tab_r.set_index(["exp", "trecho"])
    tc_p = tab_p.pivot(index="exp", columns="pos", values="t_centroide_s")
    X_CM = {p: SENSORES[p]["d"] for p in SENSORES}

    # instante de injeção estimado, em s após o minuto anotado
    T0 = {}
    for e in EXPERIMENTOS.exp:
        U_AC = (X_CM["C"] - X_CM["A"]) / 100 / (tc_p.loc[e, "C"] - tc_p.loc[e, "A"])
        T0[e] = float(round(tc_p.loc[e, "A"] - X_CM["A"] / 100 / U_AC))


    def curva_conc(exp, p):
        """NaCl em excesso no pulso (mg/L), média móvel de 5 s, zerada fora da janela do pulso;
        tempo em s desde a injeção estimada."""
        e = EXPERIMENTOS.set_index("exp").loc[exp]
        r = pulsos[(exp, p)]
        serie = nacl_total[p].loc[e.injecao - pd.Timedelta(seconds=60):e.fim].interpolate(
            limit_direction="both").values
        c = pd.Series(serie - np.nanmedian(serie[:60])).rolling(5, center=True, min_periods=1).mean().values
        c = np.clip(c, 0, None)
        c[(r["t"] < r["t_ini"]) | (r["t"] > r["t_fim"])] = 0.0
        return r["t"] - T0[exp], c


    # tempos característicos da pluma em cada sensor
    tempos = pd.DataFrame([dict(exp=e, pos=p, d_cm=X_CM[p], t0_s=T0[e],
                                chegada_s=r["t_ini"] - T0[e], pico_s=r["t_pico"] - T0[e],
                                centroide_s=r["tm"] - T0[e], saida_s=r["t_fim"] - T0[e],
                                duracao_s=r["t_fim"] - r["t_ini"])
                           for (e, p), r in pulsos.items()]).sort_values(["exp", "pos"])
    tempos.round(1).to_csv(PASTA_TAB / "pluma_tempos.csv", index=False)

    # velocidades da frente, do pico, do centroide e da cauda entre A e E
    vel = []
    for e in EXPERIMENTOS.exp:
        a = tempos[(tempos.exp == e) & (tempos.pos == "A")].iloc[0]
        z = tempos[(tempos.exp == e) & (tempos.pos == "E")].iloc[0]
        dx = z.d_cm - a.d_cm
        vel.append(dict(exp=e, U_frente_cm_s=dx / (z.chegada_s - a.chegada_s),
                        U_pico_cm_s=dx / (z.pico_s - a.pico_s),
                        U_centroide_cm_s=dx / (z.centroide_s - a.centroide_s),
                        U_cauda_cm_s=dx / (z.saida_s - a.saida_s),
                        alongamento_s=(z.saida_s - z.chegada_s) - (a.saida_s - a.chegada_s)))
    velocidades = pd.DataFrame(vel)
    velocidades.round(2).to_csv(PASTA_TAB / "pluma_velocidades_A_E.csv", index=False)
    display(tempos[tempos.pos.isin(["A", "C", "E"])].round(1))
    display(velocidades.round(2))

In [ ]:
# F13 – diagrama distância × tempo do ensaio em foco
if CAL_OK:
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    g = tempos[tempos.exp == ENSAIO_FOCO].set_index("pos")
    cam = g.loc[["A", "C", "E"]]
    ax.fill_betweenx(cam.d_cm, cam.chegada_s, cam.saida_s, color=CORES["A"], alpha=0.12, lw=0,
                     label="Pluma em A, C e E (da chegada à saída)")
    ax.plot(cam.chegada_s, cam.d_cm, color=CORES["A"], lw=0.8, ls="--", label="Frente")
    ax.plot(cam.saida_s, cam.d_cm, color=CORES["A"], lw=0.8, ls="-.", label="Cauda")
    ax.plot(cam.centroide_s, cam.d_cm, color=CORES["A"], lw=1.4, marker="o", ms=5, label="Centroide")
    for p in ["B", "D"]:
        ax.plot([g.loc[p, "chegada_s"], g.loc[p, "saida_s"]], [g.loc[p, "d_cm"]] * 2, color=CORES[p], lw=3,
                alpha=0.8, solid_capstyle="butt", label=f"Passagem em {p} (outra altura)")
        ax.plot(g.loc[p, "centroide_s"], g.loc[p, "d_cm"], marker="o", ms=4, color=CORES[p], mec=TINTA, mew=0.5)
    ax.plot(g.pico_s, g.d_cm, color=TINTA, lw=0, marker="x", ms=5, label="Pico")
    U = rot.loc[(ENSAIO_FOCO, TRECHO_AC), "U_rout"] * 100
    tt = np.array([0, cam.saida_s.max()])
    ax.plot(tt, U * tt, color=TINTA2, lw=0.7, ls=":", label=f"x = U t (U = {U:.1f} cm/s, A→C)".replace(".", ","))
    for p in SENSORES:
        ax.text(cam.saida_s.max() + 2 if p in "ACE" else g.loc[p, "saida_s"] + 2,
                g.loc[p, "d_cm"] + (5 if p in "BD" else -1), p, fontsize=8, color=CORES[p], va="center",
                fontweight="bold")
    v = velocidades.set_index("exp").loc[ENSAIO_FOCO]
    ax.text(0.02, 0.97, "Velocidades de A a E:\n" + f"frente {v.U_frente_cm_s:.1f} cm/s\ncentroide {v.U_centroide_cm_s:.1f} cm/s\n"
            f"cauda {v.U_cauda_cm_s:.1f} cm/s".replace(".", ","), transform=ax.transAxes, va="top", fontsize=7,
            color=TINTA2)
    ax.set_title(NOME_FOCO, fontsize=9)
    ax.set_xlabel("Tempo desde a injeção (s)")
    ax.set_ylabel("Distância ao lançamento (cm)")
    ax.set_xlim(0, cam.saida_s.max() + 12)
    ax.set_ylim(0, 245)
    ax.legend(fontsize=6.5, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.15))
    salvar(fig, "F13_diagrama_distancia_tempo")

In [ ]:
# Previsão da curva em E a partir de A (validação; tabela com os cinco ensaios) e F14 do ensaio em foco
if CAL_OK:
    linhas_prev, curvas_prev = [], {}
    dx_AE = (SENSORES["E"]["d"] - SENSORES["A"]["d"]) / 100
    for e in EXPERIMENTOS.exp:
        t, cA = curva_conc(e, "A")
        _, cE = curva_conc(e, "E")
        U, D = rot.loc[(e, TRECHO_AC), "U_rout"], rot.loc[(e, TRECHO_AC), "D_rout"]
        prevE = routing(t, cA, U, D, dx_AE)
        curvas_prev[e] = (t, cA, cE, prevE)
        nE, nP = cE / trapz(cE, t), prevE / trapz(prevE, t)
        m = (nE > 0.01 * nE.max()) | (nP > 0.01 * nP.max())
        linhas_prev.append(dict(
            exp=e, t_pico_med_s=t[np.argmax(cE)], t_pico_prev_s=t[np.argmax(prevE)],
            t_centroide_med_s=trapz(t * cE, t) / trapz(cE, t), t_centroide_prev_s=trapz(t * prevE, t) / trapz(prevE, t),
            C_pico_med_mg_L=cE.max(), C_pico_prev_mg_L=prevE.max(), razao_areas_E_A=trapz(cE, t) / trapz(cA, t),
            R2_forma=1 - np.sum((nP[m] - nE[m]) ** 2) / np.sum((nE[m] - nE[m].mean()) ** 2)))
    previsao = pd.DataFrame(linhas_prev)
    previsao["erro_t_centroide_s"] = previsao.t_centroide_prev_s - previsao.t_centroide_med_s
    previsao.round(3).to_csv(PASTA_TAB / "previsao_E_a_partir_de_A.csv", index=False)
    display(previsao.round(2))

    # F14 – (a) perfis ao longo do canal; (b) curva em E medida e prevista
    X_GRADE = np.arange(SENSORES["A"]["d"], 233.0 + 0.1, 2.0)
    t, cA = curva_conc(ENSAIO_FOCO, "A")
    U, D = rot.loc[(ENSAIO_FOCO, TRECHO_AC), "U_rout"], rot.loc[(ENSAIO_FOCO, TRECHO_AC), "D_rout"]
    campo = np.array([cA if x == X_GRADE[0] else routing(t, cA, U, D, (x - X_GRADE[0]) / 100) for x in X_GRADE])
    tm_A = tc_p.loc[ENSAIO_FOCO, "A"] - T0[ENSAIO_FOCO]
    instantes = [tm_A + (xc - X_GRADE[0]) / (100 * U) for xc in (X_GRADE[0], 146.0, 190.0, 233.0)]
    linhas_perfil = []
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 3.3), gridspec_kw={"width_ratios": [1.25, 1]})
    ax = axs[0]
    for tk, cor in zip(instantes, ["#a9c8ef", "#5b9be0", "#2a78d6", "#123f75"]):
        j = int(np.argmin(np.abs(t - tk)))
        ax.plot(X_GRADE, campo[:, j], color=cor, lw=1.3, label=f"t = {t[j]:.0f} s")
        for p, mk in [("C", "s"), ("E", "D")]:
            tp, cp = curva_conc(ENSAIO_FOCO, p)
            ax.plot(SENSORES[p]["d"], np.interp(t[j], tp, cp), marker=mk, color=cor, ms=5, mec=TINTA, mew=0.5,
                    ls="none")
            linhas_perfil.append(dict(exp=ENSAIO_FOCO, t_s=t[j], sensor=p, medido_mg_L=np.interp(t[j], tp, cp),
                                      modelo_mg_L=float(np.interp(SENSORES[p]["d"], X_GRADE, campo[:, j]))))
    for p in ["A", "C", "E"]:
        ax.axvline(SENSORES[p]["d"], color=GRADE, lw=0.8, zorder=0)
        ax.text(SENSORES[p]["d"], 1.0, p, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=7, color=TINTA2)
    ax.plot([], [], marker="s", ls="none", color=TINTA2, mec=TINTA, mew=0.5, label="C medido")
    ax.plot([], [], marker="D", ls="none", color=TINTA2, mec=TINTA, mew=0.5, label="E medido")
    ax.set_title("(a) Pluma ao longo do canal (linhas: modelo)", loc="left", fontsize=8.5, pad=12)
    ax.set_xlabel("Distância ao lançamento (cm)")
    ax.set_ylabel("NaCl em excesso (mg/L)")
    ax.set_ylim(0, None)
    ax.legend(fontsize=6.5, ncol=2, loc="upper right")
    ax = axs[1]
    t, cA, cE, prevE = curvas_prev[ENSAIO_FOCO]
    rr = previsao.set_index("exp").loc[ENSAIO_FOCO]
    ax.plot(t, cA, color=CORES["A"], lw=1.0, label="A medido")
    ax.plot(t, cE, color=CORES["E"], lw=1.6, label="E medido")
    ax.plot(t, prevE, color=TINTA, lw=1.0, ls="--", label="E previsto a partir de A")
    ax.set_xlim(max(0, t[cA > 0].min() - 10), t[cE > 0].max() + 15)
    ax.set_title("(b) Curva em E: medida e prevista\n" + f"R² (forma) = {rr.R2_forma:.2f}".replace(".", ","),
                 loc="left", fontsize=8.5)
    ax.set_xlabel("Tempo desde a injeção (s)")
    ax.legend(fontsize=6.5, loc="upper right")
    fig.suptitle(NOME_FOCO, fontsize=9)
    fig.tight_layout()
    salvar(fig, "F14_pluma_modelo")
    pd.DataFrame(linhas_perfil).round(2).to_csv(PASTA_TAB / "pluma_perfis_conferencia.csv", index=False)

## 11. Concentração de sal e limite de cloreto (CONAMA 357/2005)

Cloreto = NaCl × 35,45/58,44. O limite de cloreto total em águas doces das classes 1, 2 e 3 da Resolução CONAMA 357/2005 é 250 mg/L Cl. O cloreto calculado é só o **adicionado** pelo sal (o da água do canal não foi medido). Diluição no pico: $S = C_{inj}/C_{pico}$, com 128 000 mg/L de NaCl na solução injetada (para atender ao limite, a solução precisa de diluição de cerca de 310 vezes).

In [ ]:
if CAL_OK:
    C_INJ = 128_000.0               # mg/L de NaCl na solução injetada
    FRACAO_CL = 35.453 / 58.443     # fração mássica de cloreto no NaCl
    LIMITE_CL = 250.0               # mg/L Cl, CONAMA 357/2005 (classes 1 a 3)
    cl_total = FRACAO_CL * nacl_total
    linhas_cl = []
    for _, e in EXPERIMENTOS.iterrows():
        jan = cl_total.loc[e.injecao:e.fim]
        for p in SENSORES:
            r = pulsos[(e.exp, p)]
            s = jan[p].dropna()
            ini = e.injecao + pd.Timedelta(seconds=float(r["t_ini"]))
            fim = e.injecao + pd.Timedelta(seconds=float(r["t_fim"]))
            base = np.nanmedian(nacl_total[p].loc[e.injecao - pd.Timedelta(seconds=60):e.injecao].values)
            pico = nacl_total[p].loc[ini:fim].max() - base
            dk_max = (kappa_A[p].loc[e.injecao:e.fim] - KAPPA_REF_A).max()
            linhas_cl.append(dict(exp=e.exp, pos=p, cl_max_mg_L=s.max(), cl_base_antes_mg_L=FRACAO_CL * base,
                                  segundos_acima_limite=int((s > LIMITE_CL).sum()),
                                  nacl_pico_pulso_mg_L=pico, diluicao_no_pico=C_INJ / pico,
                                  dk_max_escala_A=dk_max, fora_da_calibracao=bool(dk_max > DK_MAX_CAL)))
    cloreto = pd.DataFrame(linhas_cl)
    cloreto.round(2).to_csv(PASTA_TAB / "cloreto_conama.csv", index=False)
    cl_fim = cl_total.iloc[-120:-60].median()       # 1 min antes do fim do registro
    cl_fim.round(1).to_csv(PASTA_TAB / "cloreto_acumulado_fim.csv")
    display(cloreto.pivot(index="exp", columns="pos", values="cl_max_mg_L").round(0))
    display(cloreto[cloreto.segundos_acima_limite > 0])
    print("Cloreto adicionado no fim do registro (mg/L):", cl_fim.round(1).to_dict())

In [ ]:
# F16 – cloreto adicionado em todos os ensaios, com o limite da CONAMA 357:
#       (a) registro completo; (b) máximo de cada ensaio, separando o cloreto acumulado antes do pulso
if CAL_OK:
    fig, axs = plt.subplots(2, 1, figsize=(7.2, 6.4))
    ax = axs[0]
    for p in SENSORES:
        ax.plot(cl_total.index, cl_total[p], label=rotulo(p), **estilo(p))
    for _, e in EXPERIMENTOS.iterrows():
        ax.axvline(e.injecao, color=TINTA2, lw=0.6, ls="--", zorder=1)
        ax.text(e.injecao, 1.01, f"Ens. {e.exp}\n{str(e.inclinacao_graus).replace('.', ',')}°",
                transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=7, color=TINTA2)
    ax.axhline(LIMITE_CL, color="#e34948", lw=1.0, ls="--", label="Limite CONAMA 357")
    ax.set_ylim(0, 480)
    ax.set_ylabel("Cloreto adicionado (mg/L)")
    ax.set_xlabel("Horário (09/11/2026)")
    ax.xaxis.set_major_formatter(FMT_HORA)
    ax.set_title("(a) Registro completo", loc="left", fontsize=9, pad=24)
    ax.legend(ncol=3, fontsize=6.5, loc="upper left", handlelength=1.5, columnspacing=1.0)
    ax = axs[1]
    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=False)
    larg = 0.16
    for k, p in enumerate(SENSORES):
        g = cloreto[cloreto.pos == p].set_index("exp").loc[EXPERIMENTOS.exp]
        x = np.arange(len(g)) + (k - 2) * larg
        ax.bar(x, g.cl_base_antes_mg_L, width=larg * 0.88, color=CORES[p], alpha=0.3, lw=0)
        ax.bar(x, g.cl_max_mg_L - g.cl_base_antes_mg_L, bottom=g.cl_base_antes_mg_L, width=larg * 0.88,
               color=CORES[p], lw=0, label=p)
    ax.axhline(LIMITE_CL, color="#e34948", lw=1.0, ls="--")
    ax.text(len(EXPERIMENTOS) - 0.5, LIMITE_CL + 6, "250 mg/L", ha="right", va="bottom", fontsize=7, color="#e34948")
    ax.set_xticks(np.arange(len(EXPERIMENTOS)))
    ax.set_xticklabels([f"Ensaio {e.exp}\n{str(e.inclinacao_graus).replace('.', ',')}°" for _, e in EXPERIMENTOS.iterrows()])
    ax.set_ylabel("Cloreto máximo (mg/L)")
    ax.set_ylim(0, 480)
    ax.set_title("(b) Máximo em cada sensor (parte clara: cloreto acumulado antes do pulso)", loc="left", fontsize=9)
    ax.legend(ncol=5, fontsize=7, loc="upper left", title="Sensor", title_fontsize=7)
    fig.tight_layout()
    salvar(fig, "F16_cloreto_conama")

## 12. Baixar figuras e tabelas

Compacta a pasta `resultados` e baixa o `.zip` (no Colab).

In [ ]:
shutil.make_archive("resultados_G2", "zip", PASTA)
print("Arquivos gerados:")
for arq in sorted(PASTA.rglob("*.*")):
    print("  ", arq)

try:
    from google.colab import files
    files.download("resultados_G2.zip")
except ImportError:
    print("\nFora do Colab: o arquivo resultados_G2.zip está na pasta do notebook.")